# 🏥 CKD Detection: Explainable ML with Feature Selection + SHAP
### ICEFronT 2026 — MBSTU
**Dataset:** UCI Chronic Kidney Disease (Rubini, 2015) | 397 samples | 24 features

**Novel Contributions:**
1. Multi-method Feature Selection (Chi-Square + RFE + Mutual Information)
2. Ensemble ML Classification (7 models + Soft Voting)
3. SHAP Explainability — *why* the model makes each prediction

---

In [ ]:
# ============================================================
# 1. INSTALL & IMPORT LIBRARIES
# ============================================================
!pip install shap -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               HistGradientBoostingClassifier, VotingClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import (SelectKBest, chi2, RFE,
                                        mutual_info_classif, SelectFromModel)
from sklearn.utils import resample

print('All libraries imported successfully ✅')

## 📥 Step 1: Load & Parse Dataset

In [ ]:
# ============================================================
# 2. LOAD & PARSE ARFF
# ============================================================
def load_ckd_arff(filepath):
    col_names = ['age','bp','sg','al','su','rbc','pc','pcc','ba','bgr','bu','sc',
                 'sod','pot','hemo','pcv','wbcc','rbcc','htn','dm','cad',
                 'appet','pe','ane','class']
    rows = []
    in_data = False
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line.lower() == '@data':
                in_data = True
                continue
            if in_data and line and not line.startswith('%'):
                parts = [p.strip() for p in line.split(',')]
                if len(parts) == 25:
                    rows.append(parts)
    df = pd.DataFrame(rows, columns=col_names)
    df = df.replace('?', np.nan)
    return df

df = load_ckd_arff('chronic_kidney_disease.arff')

print('=' * 60)
print('UCI CKD DATASET OVERVIEW')
print('=' * 60)
print(f'Shape            : {df.shape}')
print(f'CKD cases        : {(df["class"]=="ckd").sum()}')
print(f'Non-CKD cases    : {(df["class"]=="notckd").sum()}')
print(f'Total missing    : {df.isnull().sum().sum()}')
print(f'\nMissing per feature:')
print(df.isnull().sum()[df.isnull().sum()>0])

## 🔧 Step 2: Preprocessing

In [ ]:
# ============================================================
# 3. PREPROCESSING
# ============================================================
TARGET = 'class'
X_raw = df.drop(columns=[TARGET]).copy()
y = (df[TARGET] == 'ckd').astype(int).values  # 1=CKD, 0=notCKD

NUMERIC_COLS = ['age','bp','sg','al','su','bgr','bu','sc','sod','pot',
                'hemo','pcv','wbcc','rbcc']
CATEG_COLS   = ['rbc','pc','pcc','ba','htn','dm','cad','appet','pe','ane']

# Full feature names for display
FEAT_NAMES = {
    'age':'Age','bp':'Blood Pressure','sg':'Specific Gravity',
    'al':'Albumin','su':'Sugar','rbc':'Red Blood Cells',
    'pc':'Pus Cells','pcc':'Pus Cell Clumps','ba':'Bacteria',
    'bgr':'Blood Glucose','bu':'Blood Urea','sc':'Serum Creatinine',
    'sod':'Sodium','pot':'Potassium','hemo':'Hemoglobin',
    'pcv':'Packed Cell Volume','wbcc':'WBC Count','rbcc':'RBC Count',
    'htn':'Hypertension','dm':'Diabetes Mellitus','cad':'Coronary Artery Disease',
    'appet':'Appetite','pe':'Pedal Edema','ane':'Anemia'
}

# Convert numeric
for col in NUMERIC_COLS:
    X_raw[col] = pd.to_numeric(X_raw[col], errors='coerce')

# Impute
num_imp = SimpleImputer(strategy='median')
cat_imp = SimpleImputer(strategy='most_frequent')
X_raw[NUMERIC_COLS] = num_imp.fit_transform(X_raw[NUMERIC_COLS])
X_raw[CATEG_COLS]   = cat_imp.fit_transform(X_raw[CATEG_COLS])

# Encode categorical
le_dict = {}
for col in CATEG_COLS:
    le = LabelEncoder()
    X_raw[col] = le.fit_transform(X_raw[col].astype(str))
    le_dict[col] = le

print(f'Preprocessed X shape : {X_raw.shape}')
print(f'CKD: {y.sum()} | Not CKD: {(y==0).sum()}')

## 🎯 Step 3: Feature Selection (3 Methods)

In [ ]:
# ============================================================
# 4. FEATURE SELECTION — 3 METHODS
# ============================================================
# MinMax scale for Chi-Square (needs non-negative values)
mm_scaler = MinMaxScaler()
X_mm = mm_scaler.fit_transform(X_raw)

K = 12  # Select top-K features

# Method 1: Chi-Square
chi2_sel = SelectKBest(chi2, k=K)
chi2_sel.fit(X_mm, y)
chi2_scores = pd.Series(chi2_sel.scores_, index=X_raw.columns).sort_values(ascending=False)
chi2_features = set(chi2_scores.head(K).index)

# Method 2: Mutual Information
mi_scores = mutual_info_classif(X_raw, y, random_state=42)
mi_series = pd.Series(mi_scores, index=X_raw.columns).sort_values(ascending=False)
mi_features = set(mi_series.head(K).index)

# Method 3: RFE with Random Forest
rf_rfe = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rfe = RFE(estimator=rf_rfe, n_features_to_select=K, step=1)
rfe.fit(X_raw, y)
rfe_features = set(X_raw.columns[rfe.support_])

# Consensus: features selected by at least 2 out of 3 methods
all_features = list(X_raw.columns)
vote_count = {}
for feat in all_features:
    votes = sum([
        feat in chi2_features,
        feat in mi_features,
        feat in rfe_features
    ])
    vote_count[feat] = votes

vote_df = pd.Series(vote_count).sort_values(ascending=False)
SELECTED_FEATURES = list(vote_df[vote_df >= 2].index)

print('=' * 60)
print('FEATURE SELECTION RESULTS')
print('=' * 60)
print(f'\n  Chi-Square top {K}    : {sorted(chi2_features)}')
print(f'  Mutual Info top {K}   : {sorted(mi_features)}')
print(f'  RFE top {K}           : {sorted(rfe_features)}')
print(f'\n  Consensus (≥2 votes) : {SELECTED_FEATURES}')
print(f'  Features selected    : {len(SELECTED_FEATURES)} / {X_raw.shape[1]}')

In [ ]:
# ── Feature Selection Visualization ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Feature Selection — Three Methods Comparison', fontsize=14, fontweight='bold')

# Chi-Square
top_chi2 = chi2_scores.head(K)
axes[0].barh([FEAT_NAMES.get(f,f) for f in top_chi2.index[::-1]],
             top_chi2.values[::-1], color='#1565C0')
axes[0].set_title('Chi-Square Scores')
axes[0].set_xlabel('Score')

# Mutual Information
top_mi = mi_series.head(K)
axes[1].barh([FEAT_NAMES.get(f,f) for f in top_mi.index[::-1]],
             top_mi.values[::-1], color='#2E7D32')
axes[1].set_title('Mutual Information Scores')
axes[1].set_xlabel('Score')

# Vote count heatmap-style
vote_plot = vote_df.reset_index()
vote_plot.columns = ['Feature', 'Votes']
vote_plot['FullName'] = vote_plot['Feature'].map(FEAT_NAMES)
vote_plot = vote_plot.sort_values('Votes', ascending=True)
colors_vote = ['#E53935' if v < 2 else '#43A047' for v in vote_plot['Votes']]
axes[2].barh(vote_plot['FullName'], vote_plot['Votes'], color=colors_vote)
axes[2].axvline(x=2, color='black', linestyle='--', linewidth=1.5, label='Threshold (≥2)')
axes[2].set_title('Consensus Votes (Green = Selected)')
axes[2].set_xlabel('Number of Methods Selected')
axes[2].legend()

plt.tight_layout()
plt.savefig('feature_selection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Feature selection plot saved ✅')

## ⚖️ Step 4: Split → Scale → Oversample

In [ ]:
# ============================================================
# 5. PIPELINE: Split → Scale → Oversample (train only)
# ============================================================
X_sel = X_raw[SELECTED_FEATURES]  # Use only selected features

# Split FIRST
X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
    X_sel, y, test_size=0.2, random_state=42, stratify=y
)

# Scale — fit on train ONLY
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

# Oversample minority class (train only)
def oversample_binary(X_scaled, y):
    df_tmp = pd.DataFrame(X_scaled)
    df_tmp['__y__'] = y
    majority = df_tmp[df_tmp['__y__'] == 1]
    minority = df_tmp[df_tmp['__y__'] == 0]
    minority_up = resample(minority, replace=True,
                           n_samples=len(majority), random_state=42)
    balanced = pd.concat([majority, minority_up]).sample(
        frac=1, random_state=42).reset_index(drop=True)
    return balanced.drop('__y__', axis=1).values, balanced['__y__'].values

X_train, y_train = oversample_binary(X_train_scaled, y_train_raw)

print(f'Selected features ({len(SELECTED_FEATURES)}): {[FEAT_NAMES.get(f,f) for f in SELECTED_FEATURES]}')
print(f'\nTrain (after oversample) : {X_train.shape} — CKD:{y_train.sum()} / NotCKD:{(y_train==0).sum()}')
print(f'Test  (original)         : {X_test.shape}  — CKD:{y_test.sum()} / NotCKD:{(y_test==0).sum()}')

## 🤖 Step 5: Train & Evaluate 7 Models

In [ ]:
# ============================================================
# 6. MODEL DEFINITIONS
# ============================================================
models = {
    'Decision Tree':          DecisionTreeClassifier(
                                  max_depth=8, min_samples_leaf=4,
                                  min_samples_split=8, random_state=42),
    'Random Forest':          RandomForestClassifier(
                                  n_estimators=300, max_depth=12,
                                  min_samples_leaf=2, max_features='sqrt',
                                  random_state=42, n_jobs=-1),
    'Gradient Boosting':      GradientBoostingClassifier(
                                  n_estimators=200, max_depth=4,
                                  learning_rate=0.05, subsample=0.8,
                                  random_state=42),
    'Hist Gradient Boosting': HistGradientBoostingClassifier(
                                  max_iter=300, max_depth=5,
                                  learning_rate=0.05, l2_regularization=0.1,
                                  random_state=42),
    'SVM':                    SVC(kernel='rbf', C=10, gamma='scale',
                                  probability=True, random_state=42),
    'KNN':                    KNeighborsClassifier(n_neighbors=7, weights='distance'),
    'Logistic Regression':    LogisticRegression(C=1.0, max_iter=2000,
                                                  solver='lbfgs', random_state=42),
}

# ── 10-Fold Stratified CV ─────────────────────────────────────
print('=' * 65)
print('MODEL TRAINING & EVALUATION — 10-Fold Stratified CV')
print('=' * 65)

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
results = {}
trained = {}

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train,
                                cv=skf, scoring='accuracy', n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    auc  = roc_auc_score(y_test, y_prob)

    results[name] = {
        'CV Acc (%)':    round(cv_scores.mean() * 100, 2),
        'CV Std (%)':    round(cv_scores.std()  * 100, 2),
        'Test Acc (%)':  round(acc  * 100, 2),
        'Precision (%)': round(prec * 100, 2),
        'Recall (%)':    round(rec  * 100, 2),
        'F1-Score (%)':  round(f1   * 100, 2),
        'AUC-ROC (%)':   round(auc  * 100, 2),
    }
    trained[name] = (model, y_pred, y_prob)

    print(f'\n{name}:')
    print(f'  CV  : {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')
    print(f'  Test: Acc={acc*100:.2f}%  Prec={prec*100:.2f}%  '
          f'Rec={rec*100:.2f}%  F1={f1*100:.2f}%  AUC={auc*100:.2f}%')

In [ ]:
# ── Summary Table ─────────────────────────────────────────────
results_df = (pd.DataFrame(results).T
                .reset_index()
                .rename(columns={'index': 'Model'})
                .sort_values('Test Acc (%)', ascending=False)
                .reset_index(drop=True))

print('\n' + '=' * 65)
print('PERFORMANCE SUMMARY TABLE')
print('=' * 65)
print(results_df.to_string(index=False))

## 🗳️ Step 6: Ensemble (Soft Voting)

In [ ]:
# ============================================================
# 7. ENSEMBLE SOFT VOTING
# ============================================================
ensemble = VotingClassifier(estimators=[
    ('rf',  RandomForestClassifier(n_estimators=300, max_depth=12,
                                    min_samples_leaf=2, random_state=42, n_jobs=-1)),
    ('hgb', HistGradientBoostingClassifier(max_iter=300, max_depth=5,
                                            learning_rate=0.05,
                                            l2_regularization=0.1, random_state=42)),
    ('gb',  GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                        learning_rate=0.05, subsample=0.8,
                                        random_state=42)),
], voting='soft', n_jobs=-1)

ensemble.fit(X_train, y_train)
y_pred_ens = ensemble.predict(X_test)
y_prob_ens = ensemble.predict_proba(X_test)[:, 1]

ens_acc = accuracy_score(y_test, y_pred_ens)
ens_f1  = f1_score(y_test, y_pred_ens)
ens_auc = roc_auc_score(y_test, y_prob_ens)

print('SOFT VOTING ENSEMBLE (RF + HistGB + GB)')
print('=' * 60)
print(f'Accuracy  : {ens_acc*100:.2f}%')
print(f'F1-Score  : {ens_f1*100:.2f}%')
print(f'AUC-ROC   : {ens_auc*100:.2f}%')
print('\n' + classification_report(y_test, y_pred_ens,
                                   target_names=['Not CKD', 'CKD'], digits=4))

## 📊 Step 7: Results Visualization

In [ ]:
# ============================================================
# 8. RESULTS VISUALIZATION
# ============================================================
best_name = results_df.iloc[0]['Model']
_, y_pred_best, y_prob_best = trained[best_name]

PAL = ['#1565C0','#2E7D32','#E65100','#6A1B9A','#00838F','#AD1457','#4E342E']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('CKD Detection Results — UCI Dataset (ICEFronT 2026)',
             fontsize=14, fontweight='bold')

# Plot 1: Accuracy comparison
ax1 = axes[0, 0]
all_names = list(results_df['Model']) + ['Ensemble (Voting)']
all_acc   = list(results_df['Test Acc (%)']) + [round(ens_acc*100, 2)]
bars = ax1.barh(all_names, all_acc, color=PAL + ['#B71C1C'])
ax1.set_xlabel('Test Accuracy (%)')
ax1.set_title('Model Accuracy Comparison')
ax1.set_xlim([80, 105])
for bar, val in zip(bars, all_acc):
    ax1.text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=9)

# Plot 2: Confusion Matrix
ax2 = axes[0, 1]
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2,
            xticklabels=['Not CKD','CKD'],
            yticklabels=['Not CKD','CKD'],
            annot_kws={'size':14})
ax2.set_title(f'Confusion Matrix — {best_name}')
ax2.set_xlabel('Predicted')
ax2.set_ylabel('Actual')

# Plot 3: Feature Importance (RF)
ax3 = axes[1, 0]
rf_model = trained['Random Forest'][0]
feat_imp = (pd.DataFrame({'Feature': SELECTED_FEATURES,
                           'Importance': rf_model.feature_importances_})
            .sort_values('Importance', ascending=True))
feat_imp['FullName'] = feat_imp['Feature'].map(FEAT_NAMES)
ax3.barh(feat_imp['FullName'], feat_imp['Importance'], color='#2E7D32')
ax3.set_title('Feature Importances (Random Forest)')
ax3.set_xlabel('Importance Score')

# Plot 4: ROC Curves
ax4 = axes[1, 1]
for nm, color in zip(list(results_df['Model']) + ['Ensemble (Voting)'],
                     PAL + ['#B71C1C']):
    prob = trained[nm][2] if nm != 'Ensemble (Voting)' else y_prob_ens
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_v = roc_auc_score(y_test, prob)
    ax4.plot(fpr, tpr, color=color, lw=1.5, label=f'{nm} ({auc_v:.3f})')
ax4.plot([0,1],[0,1],'k--', lw=1)
ax4.set_xlabel('False Positive Rate')
ax4.set_ylabel('True Positive Rate')
ax4.set_title('ROC Curves — All Models')
ax4.legend(fontsize=7, loc='lower right')

plt.tight_layout()
plt.savefig('ckd_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Results plot saved ✅')

## 🔍 Step 8: SHAP Explainability
**এই section টাই paper এর সবচেয়ে novel contribution।**
SHAP (SHapley Additive exPlanations) দেখায় — প্রতিটা feature model এর decision এ কতটুকু এবং কোন direction এ contribute করেছে।

In [ ]:
# ============================================================
# 9. SHAP EXPLAINABILITY — Random Forest
# ============================================================
print('Computing SHAP values... (may take ~30 seconds)')

rf_for_shap = trained['Random Forest'][0]

# TreeExplainer — fastest for tree-based models
explainer = shap.TreeExplainer(rf_for_shap)
shap_values = explainer.shap_values(X_test)

# shap_values shape: (n_samples, n_features, n_classes) or (n_samples, n_features)
# For binary: take class 1 (CKD) values
if isinstance(shap_values, list):
    shap_ckd = shap_values[1]  # class 1 = CKD
else:
    shap_ckd = shap_values

feat_display_names = [FEAT_NAMES.get(f, f) for f in SELECTED_FEATURES]
X_test_df = pd.DataFrame(X_test, columns=feat_display_names)

print('SHAP values computed ✅')
print(f'SHAP array shape: {shap_ckd.shape}')

In [ ]:
# ── SHAP Plot 1: Summary Beeswarm ─────────────────────────────
# Shows: which features matter most AND how they affect prediction
# Red = high feature value, Blue = low feature value
# Right side = pushes toward CKD, Left = pushes toward NotCKD

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_ckd, X_test_df,
                  feature_names=feat_display_names,
                  show=False, plot_size=None)
plt.title('SHAP Summary Plot — Feature Impact on CKD Prediction', 
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('SHAP summary plot saved ✅')

In [ ]:
# ── SHAP Plot 2: Bar Plot (mean |SHAP|) ────────────────────────
# Shows global feature importance based on SHAP values

plt.figure(figsize=(9, 6))
shap.summary_plot(shap_ckd, X_test_df,
                  feature_names=feat_display_names,
                  plot_type='bar', show=False, plot_size=None)
plt.title('SHAP Feature Importance (Mean |SHAP Value|)',
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('SHAP bar plot saved ✅')

In [ ]:
# ── SHAP Plot 3: Waterfall — Single Patient Explanation ────────
# Explains ONE specific prediction in detail
# Perfect for clinical interpretation in paper

# Find a correctly predicted CKD patient
ckd_indices = np.where((y_test == 1) & (trained['Random Forest'][1] == 1))[0]
patient_idx = ckd_indices[0]  # First correctly predicted CKD patient

shap_exp = shap.Explanation(
    values       = shap_ckd[patient_idx],
    base_values  = explainer.expected_value[1] if isinstance(explainer.expected_value, list)
                   else explainer.expected_value,
    data         = X_test_df.iloc[patient_idx].values,
    feature_names= feat_display_names
)

plt.figure(figsize=(10, 6))
shap.waterfall_plot(shap_exp, show=False, max_display=12)
plt.title(f'SHAP Waterfall — Patient #{patient_idx+1} (Predicted: CKD ✓)',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Waterfall plot saved — Patient #{patient_idx+1} explanation ✅')

In [ ]:
# ── SHAP Plot 4: Dependence Plot — Top 2 Features ──────────────
# Shows how top feature interacts with second feature

mean_shap = np.abs(shap_ckd).mean(axis=0)
top_feat_idx = np.argsort(mean_shap)[::-1]
top1 = feat_display_names[top_feat_idx[0]]
top2 = feat_display_names[top_feat_idx[1]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('SHAP Dependence Plots — Top 2 Features', fontsize=13, fontweight='bold')

shap.dependence_plot(top1, shap_ckd, X_test_df,
                     interaction_index=top2, ax=axes[0], show=False)
axes[0].set_title(f'{top1} vs {top2}')

shap.dependence_plot(top2, shap_ckd, X_test_df,
                     interaction_index=top1, ax=axes[1], show=False)
axes[1].set_title(f'{top2} vs {top1}')

plt.tight_layout()
plt.savefig('shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dependence plot saved ✅')

## 📋 Step 9: Final Summary for Paper

In [ ]:
# ============================================================
# 10. FINAL PAPER-READY SUMMARY
# ============================================================
print('=' * 65)
print('PAPER-READY SUMMARY — ICEFronT 2026')
print('=' * 65)

print(f'\n📌 Dataset: UCI CKD | {len(df)} samples | {X_raw.shape[1]} features')
print(f'📌 After Feature Selection: {len(SELECTED_FEATURES)} features selected')
print(f'   Selected: {[FEAT_NAMES.get(f,f) for f in SELECTED_FEATURES]}')

print(f'\n📊 BEST MODEL: {results_df.iloc[0]["Model"]}')
print(f'   Accuracy  : {results_df.iloc[0]["Test Acc (%)"]:.2f}%')
print(f'   Precision : {results_df.iloc[0]["Precision (%)"]:.2f}%')
print(f'   Recall    : {results_df.iloc[0]["Recall (%)"]:.2f}%')
print(f'   F1-Score  : {results_df.iloc[0]["F1-Score (%)"]:.2f}%')
print(f'   AUC-ROC   : {results_df.iloc[0]["AUC-ROC (%)"]:.2f}%')

print(f'\n🗳️  ENSEMBLE (Soft Voting):')
print(f'   Accuracy  : {ens_acc*100:.2f}%')
print(f'   F1-Score  : {ens_f1*100:.2f}%')
print(f'   AUC-ROC   : {ens_auc*100:.2f}%')

# Top SHAP features
mean_shap_imp = pd.Series(np.abs(shap_ckd).mean(axis=0),
                           index=[FEAT_NAMES.get(f,f) for f in SELECTED_FEATURES])
mean_shap_imp = mean_shap_imp.sort_values(ascending=False)
print(f'\n🔍 TOP SHAP FEATURES (Clinical Insights):')
for feat, val in mean_shap_imp.head(5).items():
    print(f'   {feat:<30}: {val:.4f}')

print(f'\n📁 Files generated:')
print(f'   feature_selection.png — for Section III')
print(f'   ckd_results.png       — for Section IV (Results)')
print(f'   shap_summary.png      — for Section V (Explainability)')
print(f'   shap_bar.png          — for Section V')
print(f'   shap_waterfall.png    — for Section V (Case Study)')
print(f'   shap_dependence.png   — for Section V')

print('\n' + '=' * 65)
print('COMPLETE ✅  All results are publication-ready')
print('=' * 65)